## Inspecting Stimulus Length
Loading `stimulus_fspatial_2.2.npz` to check the total number of frames generated:

In [58]:
import numpy as np
import config

data = np.load('stimulus_orientation_44.npz')
frames = data['frames']
fps = config.MC_CONFIG['fps']
print(f'Loaded {len(frames)} frames.')
print(f'At {fps} FPS, the duration is {len(frames)/fps:.2f} seconds.')


Loaded 200 frames.
At 60 FPS, the duration is 3.33 seconds.


In [59]:
import sys
import os
# Add the parent directory to sys.path so we can import dyntex
sys.path.append(os.path.abspath('..'))

import numpy as np
import imageio
import torch as tch
from dyntex.MotionCloud import MotionCloud
import config


In [60]:
def generate_motion_cloud(
    th=0.0,  # Fixed orientation
    out_path='stimulus.npz',
    duration_sec=config.LOOP_CONFIG['duration_sec'],
    N=config.MC_CONFIG['N'],
    fps=config.MC_CONFIG['fps'],
    contrast=config.MC_CONFIG['contrast'],
    ave_lum=config.MC_CONFIG['ave_lum'],
    over_samp=config.MC_CONFIG['over_samp'],
    kernel_type=config.MC_CONFIG['kernel_type'],
    sf=config.MC_CONFIG['sf'],
    sf_sig=config.MC_CONFIG['sf_sig'],
    th_sig=config.MC_CONFIG['th_sig'],
    tf=config.MC_CONFIG['tf'],
    spd_scalar=config.MC_CONFIG['spd_scalar'],
    spd_dir=config.MC_CONFIG['spd_dir'],
    octa=config.MC_CONFIG['octa']
):
    """
    Generate a single Motion Cloud with the specified parameters.
    All parameters default to those defined in config.py.
    """
    # 1) Initialize MotionCloud
    MC = MotionCloud(
        N=N,
        frame_per_second=fps,
        contrast=contrast,
        ave_lum=ave_lum,
        over_samp=over_samp,
        verbose=0
    )
    
    # 2) Set parameters
    MC.set_all(
        kernel_type=kernel_type,
        sf=sf,
        sf_sig=sf_sig,
        th=th,
        th_sig=th_sig,
        tf=tf,
        spd_scalar=spd_scalar,
        spd_dir=spd_dir,
        octa=octa
    )
    
    MC.burnout() # warm-up the AR recursion
    
    # 3) Setup video writing
    n_frames = int(fps * duration_sec)
    frames = []
    
    # 4) Generate frames & write
    for i in range(n_frames):
        MC.set_fourier_translation()
        im = MC.get_frame(adjust=True)
        arr = im.detach().cpu().numpy()
        
        # Convert to unit8
        if arr.max() > 255 or arr.min() < 0:
            arr = np.clip(arr, 0, 255)
        frame_uint8 = arr.astype(np.uint8)
        frames.append(frame_uint8)
        
    np.savez(out_path, frames=np.array(frames))
    print(f'Wrote {n_frames} frames -> {out_path}')


#### Orientation (theta)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Orientation (theta) by the step size
    val_current = loop_cfg.get('th_start', 0.0) + i * loop_cfg.get('th_step', 45.0)
    out_file = f'output/motioncloud_th_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Orientation (theta) {val_current}...')
    generate_motion_cloud(
        th=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Orientation (theta) 0.0...
Wrote 480 frames -> output/motioncloud_th_0.0.npz
Generating MotionCloud 2/6 with Orientation (theta) 45.0...
Wrote 480 frames -> output/motioncloud_th_45.0.npz
Generating MotionCloud 3/6 with Orientation (theta) 90.0...
Wrote 480 frames -> output/motioncloud_th_90.0.npz
Generating MotionCloud 4/6 with Orientation (theta) 135.0...
Wrote 480 frames -> output/motioncloud_th_135.0.npz
Generating MotionCloud 5/6 with Orientation (theta) 180.0...
Wrote 480 frames -> output/motioncloud_th_180.0.npz
Generating MotionCloud 6/6 with Orientation (theta) 225.0...


#### Spatial Frequency (sf)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Spatial Frequency (sf) by the step size
    val_current = loop_cfg.get('sf_start', 1.0) + i * loop_cfg.get('sf_step', 1.0)
    out_file = f'output/motioncloud_sf_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Spatial Frequency (sf) {val_current}...')
    generate_motion_cloud(
        sf=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Spatial Frequency (sf) 1.0...
Wrote 480 frames -> output/motioncloud_sf_1.0.npz
Generating MotionCloud 2/6 with Spatial Frequency (sf) 2.0...
Wrote 480 frames -> output/motioncloud_sf_2.0.npz
Generating MotionCloud 3/6 with Spatial Frequency (sf) 3.0...
Wrote 480 frames -> output/motioncloud_sf_3.0.npz
Generating MotionCloud 4/6 with Spatial Frequency (sf) 4.0...
Wrote 480 frames -> output/motioncloud_sf_4.0.npz
Generating MotionCloud 5/6 with Spatial Frequency (sf) 5.0...
Wrote 480 frames -> output/motioncloud_sf_5.0.npz
Generating MotionCloud 6/6 with Spatial Frequency (sf) 6.0...
Wrote 480 frames -> output/motioncloud_sf_6.0.npz

All generation tasks complete!


#### Spatial Frequency Bandwidth (sf_sig)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Spatial Frequency Bandwidth (sf_sig) by the step size
    val_current = loop_cfg.get('sf_sig_start', 0.5) + i * loop_cfg.get('sf_sig_step', 0.5)
    out_file = f'output/motioncloud_sf_sig_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Spatial Frequency Bandwidth (sf_sig) {val_current}...')
    generate_motion_cloud(
        sf_sig=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Spatial Frequency Bandwidth (sf_sig) 0.5...
Wrote 480 frames -> output/motioncloud_sf_sig_0.5.npz
Generating MotionCloud 2/6 with Spatial Frequency Bandwidth (sf_sig) 1.0...
Wrote 480 frames -> output/motioncloud_sf_sig_1.0.npz
Generating MotionCloud 3/6 with Spatial Frequency Bandwidth (sf_sig) 1.5...
Wrote 480 frames -> output/motioncloud_sf_sig_1.5.npz
Generating MotionCloud 4/6 with Spatial Frequency Bandwidth (sf_sig) 2.0...
Wrote 480 frames -> output/motioncloud_sf_sig_2.0.npz
Generating MotionCloud 5/6 with Spatial Frequency Bandwidth (sf_sig) 2.5...


KeyboardInterrupt: 

#### Orientation Bandwidth (th_sig)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Orientation Bandwidth (th_sig) by the step size
    val_current = loop_cfg.get('th_sig_start', 5.0) + i * loop_cfg.get('th_sig_step', 5.0)
    out_file = f'output/motioncloud_th_sig_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Orientation Bandwidth (th_sig) {val_current}...')
    generate_motion_cloud(
        th_sig=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Orientation Bandwidth (th_sig) 5.0...
Wrote 480 frames -> output/motioncloud_th_sig_5.0.npz
Generating MotionCloud 2/6 with Orientation Bandwidth (th_sig) 10.0...
Wrote 480 frames -> output/motioncloud_th_sig_10.0.npz
Generating MotionCloud 3/6 with Orientation Bandwidth (th_sig) 15.0...
Wrote 480 frames -> output/motioncloud_th_sig_15.0.npz
Generating MotionCloud 4/6 with Orientation Bandwidth (th_sig) 20.0...
Wrote 480 frames -> output/motioncloud_th_sig_20.0.npz
Generating MotionCloud 5/6 with Orientation Bandwidth (th_sig) 25.0...
Wrote 480 frames -> output/motioncloud_th_sig_25.0.npz
Generating MotionCloud 6/6 with Orientation Bandwidth (th_sig) 30.0...
Wrote 480 frames -> output/motioncloud_th_sig_30.0.npz

All generation tasks complete!


#### Temporal Frequency (tf)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Temporal Frequency (tf) by the step size
    val_current = loop_cfg.get('tf_start', 0.5) + i * loop_cfg.get('tf_step', 0.5)
    out_file = f'output/motioncloud_tf_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Temporal Frequency (tf) {val_current}...')
    generate_motion_cloud(
        tf=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Temporal Frequency (tf) 0.5...
Wrote 480 frames -> output/motioncloud_tf_0.5.npz
Generating MotionCloud 2/6 with Temporal Frequency (tf) 1.0...
Wrote 480 frames -> output/motioncloud_tf_1.0.npz
Generating MotionCloud 3/6 with Temporal Frequency (tf) 1.5...
Wrote 480 frames -> output/motioncloud_tf_1.5.npz
Generating MotionCloud 4/6 with Temporal Frequency (tf) 2.0...
Wrote 480 frames -> output/motioncloud_tf_2.0.npz
Generating MotionCloud 5/6 with Temporal Frequency (tf) 2.5...
Wrote 480 frames -> output/motioncloud_tf_2.5.npz
Generating MotionCloud 6/6 with Temporal Frequency (tf) 3.0...
Wrote 480 frames -> output/motioncloud_tf_3.0.npz

All generation tasks complete!


#### Speed (spd_scalar)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Speed (spd_scalar) by the step size
    val_current = loop_cfg.get('spd_scalar_start', 1.0) + i * loop_cfg.get('spd_scalar_step', 1.0)
    out_file = f'output/motioncloud_spd_scalar_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Speed (spd_scalar) {val_current}...')
    generate_motion_cloud(
        spd_scalar=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Speed (spd_scalar) 1.0...
Wrote 480 frames -> output/motioncloud_spd_scalar_1.0.npz
Generating MotionCloud 2/6 with Speed (spd_scalar) 2.0...
Wrote 480 frames -> output/motioncloud_spd_scalar_2.0.npz
Generating MotionCloud 3/6 with Speed (spd_scalar) 3.0...
Wrote 480 frames -> output/motioncloud_spd_scalar_3.0.npz
Generating MotionCloud 4/6 with Speed (spd_scalar) 4.0...
Wrote 480 frames -> output/motioncloud_spd_scalar_4.0.npz
Generating MotionCloud 5/6 with Speed (spd_scalar) 5.0...
Wrote 480 frames -> output/motioncloud_spd_scalar_5.0.npz
Generating MotionCloud 6/6 with Speed (spd_scalar) 6.0...
Wrote 480 frames -> output/motioncloud_spd_scalar_6.0.npz

All generation tasks complete!


#### Speed Direction (spd_dir)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Speed Direction (spd_dir) by the step size
    val_current = loop_cfg.get('spd_dir_start', 0.0) + i * loop_cfg.get('spd_dir_step', 45.0)
    out_file = f'output/motioncloud_spd_dir_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Speed Direction (spd_dir) {val_current}...')
    generate_motion_cloud(
        spd_dir=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Speed Direction (spd_dir) 0.0...
Wrote 480 frames -> output/motioncloud_spd_dir_0.0.npz
Generating MotionCloud 2/6 with Speed Direction (spd_dir) 45.0...
Wrote 480 frames -> output/motioncloud_spd_dir_45.0.npz
Generating MotionCloud 3/6 with Speed Direction (spd_dir) 90.0...
Wrote 480 frames -> output/motioncloud_spd_dir_90.0.npz
Generating MotionCloud 4/6 with Speed Direction (spd_dir) 135.0...
Wrote 480 frames -> output/motioncloud_spd_dir_135.0.npz
Generating MotionCloud 5/6 with Speed Direction (spd_dir) 180.0...
Wrote 480 frames -> output/motioncloud_spd_dir_180.0.npz
Generating MotionCloud 6/6 with Speed Direction (spd_dir) 225.0...
Wrote 480 frames -> output/motioncloud_spd_dir_225.0.npz

All generation tasks complete!


#### Contrast

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# Generate motion clouds in a loop using the config dictionary
loop_cfg = config.LOOP_CONFIG

for i in range(loop_cfg['num_clouds']):
    # Incrementally increase Contrast by the step size
    val_current = loop_cfg.get('contrast_start', 10.0) + i * loop_cfg.get('contrast_step', 10.0)
    out_file = f'output/motioncloud_contrast_{val_current:.1f}.npz'
    
    print(f'Generating MotionCloud {i+1}/{loop_cfg["num_clouds"]} with Contrast {val_current}...')
    generate_motion_cloud(
        contrast=val_current,
        out_path=out_file
    )

print('\nAll generation tasks complete!')


Generating MotionCloud 1/6 with Contrast 10.0...
Wrote 480 frames -> output/motioncloud_contrast_10.0.npz
Generating MotionCloud 2/6 with Contrast 20.0...
Wrote 480 frames -> output/motioncloud_contrast_20.0.npz
Generating MotionCloud 3/6 with Contrast 30.0...
Wrote 480 frames -> output/motioncloud_contrast_30.0.npz
Generating MotionCloud 4/6 with Contrast 40.0...
Wrote 480 frames -> output/motioncloud_contrast_40.0.npz
Generating MotionCloud 5/6 with Contrast 50.0...
Wrote 480 frames -> output/motioncloud_contrast_50.0.npz
Generating MotionCloud 6/6 with Contrast 60.0...
Wrote 480 frames -> output/motioncloud_contrast_60.0.npz

All generation tasks complete!
